In [1]:
# import findspark
# findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# from logger import get_logger

# logger = get_logger(__name__)

# SPARK_VERSION = "3.1.2"
# SCALA_VERSION = "2.12"

SPARK_VERSION = "4.0.1"
SCALA_VERSION = "2.13"

KAFKA_PACKAGE = f"org.apache.spark:spark-sql-kafka-0-10_{SCALA_VERSION}:{SPARK_VERSION}"

SAVE_LOCATION = ""
PRICE_SCHEMA = StructType(
    [
        StructField("currency", StringType(), nullable=True),
        StructField("price", DoubleType(), nullable=True),
        StructField("timestamp", TimestampType(), nullable=True),
    ]
)


In [2]:
spark = (
    SparkSession.builder.appName("Test Kafka-PySpark integration")  # type: ignore
    .master("local[*]")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/02 12:25:21 WARN Utils: Your hostname, tomek-arch, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/01/02 12:25:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/tomek2/studia/semestr_7/bigdata/projekt/crypto/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/tomek2/.ivy2.5.2/cache
The jars for the packages stored in: /home/tomek2/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-819bfdd6-aed9-4a99-a406-153905330ed5;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafk

In [3]:
kafkaStream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "crypto-prices")
    .option("kafka.group.id", "spark-speed")
    .option("startingOffsets", "latest")
    .load()
)

from pyspark.sql.functions import col, from_json, current_timestamp, current_timestamp, avg, stddev

df = (
    kafkaStream.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
    .select(col("key"), from_json(col("value"), PRICE_SCHEMA).alias("v"))
    .select("v.currency", "v.price", "v.timestamp")
)

In [5]:
from pyspark.sql.functions import min, max, count, avg, stddev, window, col



## test/nauka

In [16]:
df = spark.read.csv('./heart.csv', sep=',', encoding='utf-8', header=True, inferSchema=True)

In [60]:
df.show(n=2, truncate=False, vertical=False)
print('\n\n')
# df.printSchema()

# df.rdd.map(lambda r: r['typea']).collect()
# df.select("famhist").distinct().show()
# df.select(col("famhist")).distinct().show()
# df.filter(col("tobacco") > 30).show()
df.groupBy((col("alcohol") > 20).alias('drinks a lot?')).agg({"obesity": "mean"}).show()
# df.agg({"alcohol": "mean"}).show()

+---+-------+----+---------+-------+-----+-------+-------+---+---+
|sbp|tobacco|ldl |adiposity|famhist|typea|obesity|alcohol|age|chd|
+---+-------+----+---------+-------+-----+-------+-------+---+---+
|160|12.0   |5.73|23.11    |Present|49   |25.3   |97.2   |52 |1  |
|144|0.01   |4.41|28.61    |Absent |55   |28.87  |2.06   |63 |1  |
+---+-------+----+---------+-------+-----+-------+-------+---+---+
only showing top 2 rows



+-------------+------------------+
|drinks a lot?|      avg(obesity)|
+-------------+------------------+
|         true| 26.35199999999999|
|        false|25.917003058103994|
+-------------+------------------+



In [30]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit, current_date, to_date, minute, timestamp_diff
from pyspark.sql.types import DayTimeIntervalType

# df = DataFrame.withColumn("current_timestamp", current_timestamp())
data = [{"name": "Alice", "age": 1}]
df = spark.createDataFrame(data)
df = df.withColumn("static_date", lit("2025-12-15"))
df = df.withColumn("static_timestamp", lit("2025-12-15 15:57:50"))
df = df.withColumn("current_date", current_date())
df = df.withColumn("current_timestamp", current_timestamp())

df.show(truncate=False)

df = df.withColumn("static_date", to_date(col("static_date")))

df.select(timestamp_diff('second', 'current_timestamp', 'static_timestamp')).show()

+---+-----+-----------+-------------------+------------+--------------------------+
|age|name |static_date|static_timestamp   |current_date|current_timestamp         |
+---+-----+-----------+-------------------+------------+--------------------------+
|1  |Alice|2025-12-15 |2025-12-15 15:57:50|2025-12-15  |2025-12-15 17:04:50.006001|
+---+-----+-----------+-------------------+------------+--------------------------+

+----------------------------------------------------------+
|timestampdiff(second, current_timestamp, static_timestamp)|
+----------------------------------------------------------+
|                                                     -4021|
+----------------------------------------------------------+

